# rPPG — veri hazırlama (parça parça, Kaggle CPU)

Bu defter **yalnızca veri hazırlar**, eğitim yapmaz. 6. turda hazırlama ve eğitim aynı anda çalışırken defter 4.6 saatte
durmuştu (RAM ya da disk). Artık her parça ayrı bir oturumda hazırlanır ve sonucu Hugging Face'teki gizli
`<kullanıcı>/rppg-veri` deposuna yüklenir. Eğitim defteri veriyi oradan alır.

* İşlenen her video için etiket kalitesi (`.kal`: SNR, işaret, zaman kayması) burada hesaplanır ve birlikte yüklenir.
* Yüklenen kareler yerelde silinir, disk dolmaz.
* Oturum biterse ya da düşerse **aynı defteri tekrar çalıştırın**: depodakiler atlanır, kalınan yerden devam edilir.

**Ayarlar (bir kez):**
1. **Settings → Accelerator: None** (CPU yeterli; GPU kotası harcanmaz). **Settings → Internet: on**.
2. **Add-ons → Secrets**: `HF_TOKEN` (okuma; "public gated repos" izni) ve `HF_WRITE_TOKEN` (yazma) bu deftere bağlı olmalı.
   `dlcn_cohface` parçası için ayrıca **Add Input → `dalaoplan/rppg-dlcn`**.
   `ubfcphys` parçası için `HF_TOKEN` hesabı huggingface.co/datasets/jjuik2014/UBFC-Phys-all koşullarını kabul etmiş olmalı.
3. Aşağıdaki `PARCA` değerini seçin, **Save Version → Save & Run All**.

In [ ]:
# 0) Hangi parça hazırlanacak
PARCA = "ubfcphys"

PARCALAR = {
    # ad: (dl_prepare.py argümanları, HF'e yüklenecek alt klasörler)
    "mmpd_vipl": ("--mmpd 660 --vipl 2500 --max-sec 45 --workers 3", ["test_mmpd", "mmpd", "test_vipl", "vipl"]),
    # UBFC-Phys (56 kişi x 3 görev, bilek sensörü): HF'te 40 tek kişilik (~15 GB) + 8 çok kişilik (~30 GB) zip, ~850 GB.
    # workers 1: aynı anda tek zip diskte (8. tur için, 18 Ekim). Kişi no % 5 == 0 -> test_ubfcphys (11 kişi).
    "ubfcphys":  ("--ubfcphys 48 --max-sec 60 --workers 1",           ["ubfcphys", "test_ubfcphys"]),
    # DLCN: Add Input -> dalaoplan/rppg-dlcn gerekir. Eğitim oturumunda 43 GB'lık girdiyi okumamak için burada hazırlanır.
    "dlcn_cohface": ("--dlcn 784 --cohface 164 --max-sec 30 --workers 3",
                     ["test_dlcn", "dlcn", "test_cohface", "cohface"]),
}
PREP_ARGS, SUBS = PARCALAR[PARCA]
OTURUM_SAAT = 11.0   # Kaggle sınırı 12 saat; son yükleme için pay

In [ ]:
# 1) Ortam, kod, anahtarlar
import os, subprocess, shutil, sys, time, glob
from datetime import datetime, timedelta

T0 = datetime.now()
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"
REPO = "/kaggle/working/rppg"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

from kaggle_secrets import UserSecretsClient
for name in ("HF_TOKEN", "HF_WRITE_TOKEN"):
    try:
        os.environ[name] = UserSecretsClient().get_secret(name)
        print(f"{name} var")
    except Exception:
        raise SystemExit(f"{name} yok: Add-ons -> Secrets -> {name} kutusunu işaretleyip defteri yeniden başlatın")

from huggingface_hub import HfApi
print("Hugging Face hesabı:", HfApi(token=os.environ["HF_WRITE_TOKEN"]).whoami()["name"])

DATA, RAW, OUT = "/tmp/dl", "/tmp/dl_raw", "/kaggle/working/cikti"
for d in (DATA, RAW, OUT):
    os.makedirs(d, exist_ok=True)
import psutil
print(f"disk /tmp boş {shutil.disk_usage('/tmp').free / 1e9:.0f} GB | RAM {psutil.virtual_memory().total / 1e9:.0f} GB"
      f" | CPU {os.cpu_count()} | parça {PARCA}: {PREP_ARGS}")

In [ ]:
# 2) Depoda olanları al (kalınan yerden devam)
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")
SYNC = [PY, "-u", "scripts/dl_hf_sync.py", "--data", DATA, "--subs", *SUBS]
r = subprocess.run(SYNC + ["--pull"], capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
print(r.stdout[-3000:], r.stderr[-3000:])

In [ ]:
# 3) Hazırlama + arka planda yükleme (15 dakikada bir; hazırlama bitince son kez)
log = lambda n: open(os.path.join(OUT, n + ".log"), "a")
prep = subprocess.Popen(f"{PY} -u scripts/dl_prepare.py --out {DATA} --raw {RAW} {PREP_ARGS}".split(),
                        env=env, stdout=log("hazirla"), stderr=subprocess.STDOUT)
sync = subprocess.Popen(SYNC + ["--push", "--loop", "900", "--until-pid", str(prep.pid)],
                        env=env, stdout=log("yukle"), stderr=subprocess.STDOUT)
print("başladı:", datetime.now().strftime("%H:%M"))

In [ ]:
# 4) İzleme: 10 dakikada bir özet; süre dolarsa hazırlama durdurulur (sonraki oturum devam eder)
def tail(name, n):
    try:
        return open(os.path.join(OUT, name + ".log"), encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

def counts():
    c = {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in SUBS}
    up = sum(len(glob.glob(os.path.join(DATA, s, "*.yuklendi"))) for s in SUBS)
    return c, up

deadline = T0 + timedelta(hours=OTURUM_SAAT)
while prep.poll() is None:
    if datetime.now() > deadline:
        print("süre doldu: hazırlama durduruluyor, yüklenmeyenler son turda yüklenecek", flush=True)
        subprocess.run(["pkill", "-TERM", "-P", str(prep.pid)])
        prep.terminate()
        break
    c, up = counts()
    print(f"{time.strftime('%H:%M')} | işlenen {c} | yüklenen {up} | disk {shutil.disk_usage('/tmp').free/1e9:.0f} GB"
          f" | RAM boş {psutil.virtual_memory().available/1e9:.1f} GB | {(tail('hazirla', 1) or [''])[0][:120]}",
          flush=True)
    time.sleep(600)
prep.wait()
print("hazırlama çıkış kodu:", prep.returncode)
sync.wait()   # until-pid: hazırlama bitince son yüklemeyi yapıp çıkar
print("yükleme çıkış kodu:", sync.returncode)
print("\n".join(tail("yukle", 5)))
print("\n".join(tail("hazirla", 15)))

In [ ]:
# 5) Etiket kalitesi özeti (kareler silindi; .kal dosyalarından okunur)
r = subprocess.run([PY, "scripts/dl_quality.py", "--data", DATA, "--workers", "4", "--sources", *SUBS],
                   capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
print(r.stdout[-4000:], r.stderr[-2000:])
if os.path.exists(os.path.join(DATA, "kalite.csv")):
    shutil.copy(os.path.join(DATA, "kalite.csv"), os.path.join(OUT, f"kalite_{PARCA}.csv"))
c, up = counts()
print(f"\nişlenen {c}, yüklenen {up} | süre {datetime.now() - T0}")
print("Eksik kaldıysa defteri tekrar çalıştırın (kalınan yerden devam eder).")
os.chdir("/kaggle/working")
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın